In [ ]:
import chromadb.config

class VectorStore:
    def __init__(self,path = ".\vectorstore",collection_name ="dsa_docs"):
        self.client = chromadb.Client(chromadb.config.Settings(persist_directory=path,anonymized_telemetry=False))
        self.collection = self.client.get_or_create_collection(name=collection_name)
    
    def add_chunks(self,chunks,embed_fn):
        existing = self.collection.count()
        if existing>0:
            print("Loading_existing")
        existing_ids = set(self.collection.get()["ids"])
        new_chunks,new_embeddings,new_ids = [],[],[]

        print("Building new vector store")
        for idx,chunk in enumerate(chunks):
            id_ = str(idx)
            if id_ not in existing_ids:
                new_chunks.append(chunk)
                new_embeddings.append(embed_fn(chunk))
                new_ids.append(id_)
        
        if new_chunks:
            self.collection.add(documents=new_chunks,embeddings=new_embeddings,ids=new_ids)
            print(f"Added{len(new_chunks)} new chunks")
        
    def search(self,query_vector,top_k=3):
        result = self.collection.query(query_embeddings=[query_vector],n_results=top_k)
        return result["documents"][0] if result["documents"] else []


In [ ]:
def reciprocal_rank_fusion(bm25_chunks, chroma_chunks, k=60):
    scores = {}

    for rank, chunk in enumerate(bm25_chunks):
        scores[chunk] = scores.get(chunk, 0) + 1 / (rank + k)

    for rank, chunk in enumerate(chroma_chunks):
        scores[chunk] = scores.get(chunk, 0) + 1 / (rank + k)

    # sort by highest score
    return sorted(scores, key=scores.get, reverse=True)


In [ ]:
from rank_bm25 import BM25Okapi
import re
class BM25Search:
    def __init__(self):
        self.bm25 = None
        self.chunks = []
    def build(self,chunks):
        self.chunks = chunks
        tokenized = [re.findall(r'\b\w+\b', chunk.lower()) for chunk in chunks]
        self.bm25 = BM25Okapi(tokenized)
    def search(self, query, top_k=3):
        if self.bm25 is None:
            return []

        query_tokens = query.lower().split()
        scores = self.bm25.get_scores(query_tokens)

        top_indices = sorted(range(len(scores)),key=lambda i: scores[i],reverse=True)

        # ✅ FILTER chunks containing query words
        filtered = []
        for i in top_indices:
            chunk = self.chunks[i].lower()
            if all(q in chunk for q in query_tokens):
                filtered.append(self.chunks[i])
            if len(filtered) == top_k:
                break
        print("BM25 filtered:", filtered)
        return filtered if filtered else [self.chunks[i] for i in top_indices[:top_k]]

In [ ]:
from nltk.corpus import stopwords
from math import sqrt
import re
class DocumentLoader():
    def __init__(self,vector_store):
        self.chunks = []
        self.embedded_chunks = []
        self.text = ""
        self.vector_store = vector_store
        self.bm25 = BM25Search()
        
    # Change this line in your loader class:
    def load_file(self, path):
        with open(path, encoding="utf-8") as f:  # <--- Add encoding here
            self.text = f.read()
        return self.text
    
    def chunks_creation(self, size=500, overlap=100):
        sentences = re.split(r'(?<=[.!?]) +', self.text)
        self.chunks = []
        chunk = ""
        for sentence in sentences:
            if len(chunk.split()) + len(sentence.split()) <= size:
                chunk += " " + sentence
            else:
                self.chunks.append(chunk.strip())
                chunk = sentence
        if chunk:
            self.chunks.append(chunk.strip())   
        return self.chunks
    
    def embed_all_chunks(self,embed_fn):
        self.vector_store.add_chunks(self.chunks,embed_fn)
        self.bm25.build(self.chunks)
        
    
    def top_3_chunks(self,qurey_vector):
        return self.vector_store.search(qurey_vector,top_k=3)

In [ ]:
import ollama
from collections import OrderedDict,deque
import re

class ChatMemory:

    def __init__(self,model = "qwen2.5",system_prompt = "be a tutor",max_tokens=3000,cache_size=10,path=None):
        self.model = model
        self.max_tokens = max_tokens
        self.cache_size = cache_size
        self.chat_history = deque([{"role":"system","content":system_prompt}])
        self.cache = OrderedDict()
        self.ai_counter = 0
        self.cache_counter = 0
        self.stack = []
        self.observation_cache = {}
        
        self.vector_store = VectorStore(
            path="./vectorstore",
            collection_name="dsa_docs"
        )

        self.loader = DocumentLoader(self.vector_store)
        self.path = path
        self.doc_text = ''
        
   
    # ---------------- NORMALIZE ----------------
    def normalize(self, text):
        return re.sub(r'[^\w\s]', '', text).strip().lower()

    # ---------------- EMBEDDING ----------------
    def embed(self, text):
        return ollama.embeddings(
            model=self.model,
            prompt=text
        )["embedding"]

    # ---------------- DOCUMENT ----------------
    def prepare_document(self, path, size=1000, overlap=10):
        self.loader.load_file(path)
        self.loader.chunks_creation(size, overlap)
        self.loader.embed_all_chunks(self.embed)

    # ---------------- RETRIEVAL ----------------    
    #retriving the context where it is availabe in the doc or not
    def retrive_context(self,user_text):
        
        embedded_user = self.embed(user_text)
        chroma_chunks  = self.loader.top_3_chunks(embedded_user)
        bm25_chunks = self.loader.bm25.search(user_text, top_k=3)

        print("BM25 Chunks:", bm25_chunks)
        print("Chroma Chunks:", chroma_chunks)
        
        merged = reciprocal_rank_fusion(bm25_chunks, chroma_chunks)
        top_chunks = merged[:3]
        if not top_chunks:
            return "No Context Found"
        
        combined = " ".join (top_chunks)
        # if len(combined.split())<15:
        #     return "No Context Found"
        print("DEBUG:", top_chunks)
        return "\n\n".join(top_chunks)
    
    # ---------------- TOKEN ----------------
    def count_tokens(self, text):
        return len(text) // 4

    def total_tokens(self):
        content = " ".join(m["content"] for m in self.chat_history)
        return self.count_tokens(content)
    
    def trim_to_budget(self):
        while self.total_tokens()>self.max_tokens:
            del self.chat_history[1]
            del self.chat_history[1]
            
    # ---------------- HISTORY ----------------     
    #adding user input & reply to the history 
    def add_user(self, text):
        text = self.normalize(text)
        self.chat_history.append({"role":"user","content":text})
        self.trim_to_budget()
        return text
    
    def add_assistant(self,text):
        self.chat_history.append({"role":"assistant","content":text})
        
    def get_history(self):
        return list(self.chat_history)
    
    # ---------------- CACHE ----------------
    def check_cache(self,text):
        if text in self.cache:
            self.cache_counter+=1
            return self.cache[text]
        else: return None
    
    def store_cache(self,text,ai_reply):
        if len(self.cache)>=self.cache_size:
            self.cache.popitem(last=False)
        self.cache[text] = ai_reply
        self.ai_counter+=1
        
    def get_stats(self):
        total = self.cache_counter + self.ai_counter
        if total == 0:
            return "hit rate: 0%"
        return f"hit rate: {(self.cache_counter / total) * 100:.2f}%"
    
    # ---------------- TOOLS ----------------
    def execute_action(self, action):

        if action.startswith("search"):
            query = action[len('search("'):-2]
            return self.retrive_context(query)

        elif action.startswith("calculate"):
            expr = action[len('calculate("'):-2]
            try:
                return str(eval(expr))
            except:
                return "calculation error"

        elif action.startswith("history"):
            history = list(self.chat_history)[1:]
            return "\n".join(f"{h['role']}: {h['content']}" for h in history)

        return "Unknown action"
    
    
    def react_chat(self,user_input):
    
        context = """
        You are a ReAct agent.

        You MUST follow this format exactly:

        Thought: describe your reasoning
        Action: tool_name("input")
        Observation: result from the tool
        (repeat if needed)
        Final Answer: your final answer

        STRICT RULES:
        You MUST ALWAYS respond with Thought and Action BEFORE Final Answer.
        If you do not follow this, your response is INVALID.
        For every question, your FIRST step MUST be an Action.
        ``
        - Answer ONLY using the information present in the DOCUMENT CONTEXT.
        - You are NOT allowed to answer from your own knowledge.
        - You are FORBIDDEN from using your own knowledge.
        - You are FORBIDDEN from using “general knowledge”.
        - You MUST use tools to obtain information before answering.
        - For ANY question about algorithms, data structures, or explanations,
        you MUST first use the search action to consult the document.
        - If a question cannot be answered using tools,
        reply EXACTLY:
        "Context not available in the provided document."
        - If the user asks about conversation history,
        you MUST use the history action.
        - NEVER explain using phrases like:
        "from general knowledge"
        "based on my knowledge"
        "I know that"
        - For algorithm or data-structure questions:
        You MUST use the search action first.
        - For math questions:
        You MUST use the calculate action.
        - For conversation questions:
        You MUST use the history action.
        - Use ONE action at a time
        - Answer ONLY using the information present in the DOCUMENT CONTEXT.
        - Do NOT add new knowledge.
        - Do NOT infer beyond the document.
        - NEVER invent observation


        """
        context += f"\nUser Question: {user_input}\n"

        action_used = False
        MAX_STEPS = 10

        for _ in range(MAX_STEPS):
            response = ollama.chat(
                model=self.model,
                messages=[{"role": "user", "content": context}]
            )["message"]["content"]

            #  Validate structure (force ReAct format)
            # if not ("Thought:" in response and ("Action:" in response or "Final Answer:" in response)):
            #     context += "\nFollow the format strictly. \n"  # add this to guide the model
            #     return "Invalid format: follow ReAct structure"

            # context += response + "\n"
            # print("--------------raw response:", response)

            # ✅ Detect Action
            if "Action:" in response:
                action_used = True
                action_line = [
                    line for line in response.splitlines()
                    if line.startswith("Action:")
                ][0]
                action = action_line.replace("Action:", "").strip()
                if action in self.observation_cache:
                    observation = self.observation_cache[action]
                else:
                    observation = self.execute_action(action)
                    self.observation_cache[action] = observation
                context += f"Observation: {observation}\n"
                continue

            # ✅ Enforce rule on Final Answer
            if "Final Answer:" in response:
                if not action_used:
                    return "Context not available in the provided document."
                return response.split("Final Answer:")[-1].strip()

        return "Context not available in the provided document."
    
    # ---------------- MAIN ENTRY ----------------
    def chat(self,user_):
        normalized = self.normalize(user_)
        self.chat_history.append({"role":"user","content":user_})
        self.trim_to_budget()
        cached = self.check_cache(normalized)
        if cached:
            reply = cached
        else:
            reply = self.react_chat(user_)
            self.store_cache(normalized,reply)
            
        self.add_assistant(reply)
        return reply 
    

## agent function

In [ ]:
import math
#--------------------------------------------------------------------------------------------------------------------------------------------
#TOOL 1: search document
#--------------------------------------------------------------------------------------------------------------------------------------------
def search_document(query,memory):
    return memory.retrive_context(query)

#--------------------------------------------------------------------------------------------------------------------------------------------
#TOOL 2: calculate
#--------------------------------------------------------------------------------------------------------------------------------------------
def calculate(expression):
   try:
       allowed = "0123456789+-*/(). eE"
       cleaned = expression.replace(" ", "")
    #    print(f"DEBUG expression: {repr(cleaned)}")  # add this
       if any(ch not in allowed for ch in cleaned):
           bad = [ch for ch in cleaned if ch not in allowed]
        #    print(f"DEBUG blocked chars: {bad}")  # add this
           return "unsafe expression"
       result = eval(expression, {"__builtins__": None}, vars(math))
    #    print(f"DEBUG result: {result}")  # add this
       return str(result)
   except Exception as e:
       return f"calculation error: {str(e)}"
    
# -------------------------------
# TOOL 3: define_term
# -------------------------------
def define_term(term):
    definitions = {
        "array": "A linear data structure storing elements in contiguous memory.",
        "graph": "A set of nodes connected by edges; can be directed or undirected.",
        "queue": "A FIFO (first-in-first-out) data structure.",
        "stack": "A LIFO (last-in-first-out) data structure.",
        "tree": "A hierarchical data structure with a root and child nodes.",
    }
    term = term.lower().strip()
    return definitions.get(term, "Definition not found.")

# -------------------------------
# TOOL 4: get_history
# -------------------------------
def get_history(memory):
    history = list(memory.chat_history)[1:]
    return "\n".join(f"{m['role']}: {m['content']}" for m in history)

TOOLS = {
    "search": search_document,
    "calculate": calculate,
    "define": define_term,
    "history": get_history
}

In [ ]:
memory = ChatMemory()
memory.prepare_document(
    path=r"clean_output.md",
    size=200,
    overlap=50)
#print("BM25 result:", memory.loader.bm25.search("binary search"))
# memory.loader.embed_all_chunks(model='qwen2.5')
#print(memory.retrive_context("binary search"))
while True:
    user = input("chat: ")
    print("user :",user)
    if user.lower() == "quit":
        print(memory.get_stats())
        break
    reply = memory.chat(user)
    print("qwen2.5:", reply)

Building new vector store
Added25 new chunks
user : Why is insertion in a linked list O(1) but searching is O(n)?
BM25 filtered: []
BM25 Chunks: ['The only di®erence is\nthat each node has a reference to both the next and previous nodes in the list.\nReverse Traversal\nSingly linked lists have a forward only design, which is why the reverse traversal\nalgorithm de¯ned in x2.1.5 required some creative invention. Doubly linked lists\nmake reverse traversal as simple as forward traversal (de¯ned in x2.1.4) except\nthat we start at the tail node and update the pointers in the opposite direction.\nFigure 2.6 shows the reverse traversal algorithm in action.\nSummary\nLinked lists are good to use when you have an unknown number of items to\nstore. Using a data structure like an array would require you to specify the size\nup front; exceeding that size involves invoking a resizing algorithm which has\na linear run time.', "When\nyou use an API like that of DSA and you see a general purpose met

 Capital of India
 Time complexity answered without retrieval
 What does the document say about time complexity?
 Explain binary search from the document and calculate the maximum number of steps for 1024 elements.
 What did we discuss so far?
 Compare stack and queue from the document and tell which one is better for recursion.